# 03 — Motor de recomendación de draft
**Proyecto:** Recomendador de draft de Brawl Stars con reconocimiento visual de brawlers

**Objetivo:** a partir de las partidas preparadas en el notebook 02, calcular dos tablas estadísticas y usarlas para recomendar:
- **Aliado (sinergia):** brawlers que, jugando en el mismo equipo, tienen mayor tasa de victoria conjunta con el brawler elegido.
- **Enemigo (counter):** brawlers que, jugando contra el brawler elegido, tienen mayor tasa de victoria.

Cada recomendación se acompaña de su número de partidas y de una cota inferior de confianza, para que se pueda defender con datos.

## 1. Configuración

In [1]:
DATA_DIR = "/content/data"   # salida del notebook 02
MIN_MATCHES = 100            # partidas mínimas para considerar una estadística
TOP_K = 5                    # recomendaciones por consulta
Z = 1.96                     # nivel de confianza del 95 % para el intervalo de Wilson

## 2. Carga de datos
Usamos la tabla larga generada en el notebook 02 (una fila por brawler por partida) y la lista de los 20 brawlers del alcance inicial.

In [ ]:
import numpy as np
import pandas as pd

long = pd.read_parquet(f"{DATA_DIR}/brawl_long.parquet")
candidatos = pd.read_csv(f"{DATA_DIR}/selected_brawlers.csv")["brawler"].tolist()

requeridas = {"match_id", "map", "mode", "side", "brawler", "win"}
assert requeridas.issubset(long.columns), f"Faltan columnas: {requeridas - set(long.columns)}"

print("Filas:", len(long))
print("Brawlers en el catálogo de partidas:", long["brawler"].nunique())
print("Candidatos (alcance inicial):", len(candidatos))

## 3. Limpieza: partidas con ganador válido
Una partida es válida si exactamente un equipo tiene `win = 1`. Las partidas sin ganador claro se descartan; si se conservaran, contarían como derrota para ambos equipos y sesgarían las tasas.

In [ ]:
ganadores = long.groupby(["match_id", "side"])["win"].max().groupby("match_id").sum()
validas = ganadores[ganadores == 1].index

n_total = len(ganadores)
long = long[long["match_id"].isin(validas)].reset_index(drop=True)
print(f"Partidas válidas: {len(validas)} de {n_total} (descartadas: {n_total - len(validas)})")
print("Filas tras limpieza:", len(long))

## 4. Funciones base
- `match_arrays`: convierte la tabla larga en dos matrices (una por equipo) con los tres brawlers de cada partida y el resultado del equipo 0.
- `build_tables`: calcula la tasa de victoria base de cada brawler, la tabla de sinergias y la tabla de counters.
- `wilson_lower`: cota inferior del intervalo de Wilson. Con pocas partidas, la tasa observada es poco fiable; esta cota la castiga y permite ordenar con prudencia.

In [ ]:
def match_arrays(lg):
    t = (lg.groupby(["match_id", "side"])
           .agg(b=("brawler", list), win=("win", "max"))
           .reset_index())
    t0 = t[t["side"] == 0].set_index("match_id")
    t1 = t[t["side"] == 1].set_index("match_id")
    idx = t0.index.intersection(t1.index)
    t0, t1 = t0.loc[idx], t1.loc[idx]

    A0 = np.array(t0["b"].tolist())
    A1 = np.array(t1["b"].tolist())
    assert A0.shape[1] == 3 and A1.shape[1] == 3, "Se esperaban equipos de 3 brawlers"
    w0 = t0["win"].to_numpy().astype(int)
    return A0, A1, w0


def build_tables(lg):
    A0, A1, w0 = match_arrays(lg)

    base = lg.groupby("brawler")["win"].agg(partidas="size", victorias="sum")
    base["wr"] = base["victorias"] / base["partidas"]

    # Sinergia: pares de aliados dentro del mismo equipo
    pares = []
    for team, w in [(A0, w0), (A1, 1 - w0)]:
        for i, j in [(0, 1), (0, 2), (1, 2)]:
            pares.append(pd.DataFrame({"brawler": team[:, i], "aliado": team[:, j], "win": w}))
            pares.append(pd.DataFrame({"brawler": team[:, j], "aliado": team[:, i], "win": w}))
    syn = (pd.concat(pares, ignore_index=True)
             .groupby(["brawler", "aliado"])
             .agg(partidas=("win", "size"), victorias=("win", "sum"))
             .reset_index())

    # Counter: resultado de cada brawler frente a cada rival en el equipo contrario
    cont = []
    for i in range(3):
        for j in range(3):
            cont.append(pd.DataFrame({"brawler": A0[:, i], "rival": A1[:, j], "win": w0}))
            cont.append(pd.DataFrame({"brawler": A1[:, j], "rival": A0[:, i], "win": 1 - w0}))
    counter = (pd.concat(cont, ignore_index=True)
                 .groupby(["brawler", "rival"])
                 .agg(partidas=("win", "size"), victorias=("win", "sum"))
                 .reset_index())

    return {"base": base, "syn": syn, "counter": counter, "n_partidas": len(w0)}


def wilson_lower(victorias, partidas, z=Z):
    p = victorias / partidas
    n = partidas
    centro = p + z**2 / (2 * n)
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (centro - margen) / (1 + z**2 / n)


# Prueba rápida de la cota de Wilson: pocas partidas deben dar cotas más bajas
print("Wilson 60/100 :", round(wilson_lower(60, 100), 3))
print("Wilson 6/10   :", round(wilson_lower(6, 10), 3))
print("Wilson 600/1000:", round(wilson_lower(600, 1000), 3))

## 5. Construcción de tablas globales
Calculamos las tablas con todas las partidas válidas (sin filtro de mapa ni de modo). Es la versión por defecto del recomendador.

In [ ]:
import time
t0 = time.time()
tablas_global = build_tables(long)
print(f"Partidas usadas: {tablas_global['n_partidas']}")
print(f"Pares de sinergia: {len(tablas_global['syn'])}")
print(f"Pares de counter: {len(tablas_global['counter'])}")
print(f"Tiempo: {time.time() - t0:.1f} s")

## 6. Motor de recomendación
`recomendar` recibe el brawler elegido, el rol (`aliado` o `enemigo`) y las tablas. Aplica el mínimo de partidas, descarta al propio brawler y ordena por la cota inferior de Wilson. También devuelve la diferencia frente a la tasa base del candidato, para mostrar si la combinación es mejor de lo que normalmente rinde ese brawler.

In [ ]:
def recomendar(brawler, rol, tablas, candidatos=candidatos, top_k=TOP_K, min_partidas=MIN_MATCHES):
    base = tablas["base"]
    if rol == "aliado":
        df = tablas["syn"][tablas["syn"]["brawler"] == brawler].rename(columns={"aliado": "candidato"})
    elif rol == "enemigo":
        # candidato = brawler que gana contra el brawler elegido (rival)
        df = tablas["counter"][tablas["counter"]["rival"] == brawler].rename(columns={"brawler": "candidato"})
    else:
        raise ValueError("rol debe ser 'aliado' o 'enemigo'")

    df = df[df["candidato"].isin(candidatos) & (df["candidato"] != brawler) & (df["partidas"] >= min_partidas)].copy()
    if df.empty:
        return pd.DataFrame(columns=["candidato", "wr", "partidas", "wilson_lb", "dif_vs_base"])

    df["wr"] = df["victorias"] / df["partidas"]
    df["wilson_lb"] = wilson_lower(df["victorias"], df["partidas"])
    df["dif_vs_base"] = df["wr"] - df["candidato"].map(base["wr"])

    out = df.sort_values("wilson_lb", ascending=False).head(top_k)
    return out[["candidato", "wr", "partidas", "wilson_lb", "dif_vs_base"]].reset_index(drop=True).round(3)

## 7. Validación independiente
Comprobamos una combinación contando directamente sobre la tabla de partidas, sin usar las tablas precalculadas. Si los números no coinciden, hay un error en la lógica.

In [ ]:
def validar_par(lg, a, b):
    equipos = (lg.groupby(["match_id", "side"])
                 .agg(b=("brawler", list), win=("win", "max"))
                 .reset_index())
    # Sinergia: a y b en el mismo equipo
    en_mismo = equipos[equipos["b"].apply(lambda x: a in x and b in x)]
    s_partidas, s_victorias = len(en_mismo), int(en_mismo["win"].sum())

    # Counter: a en un equipo y b en el contrario
    m = equipos.pivot(index="match_id", columns="side", values="b")
    w = equipos.pivot(index="match_id", columns="side", values="win")
    frente = m[m.apply(lambda r: (a in r[0] and b in r[1]) or (a in r[1] and b in r[0]), axis=1)]
    c_partidas = len(frente)
    c_victorias = int(sum(
        w.loc[i, 0] if a in m.loc[i, 0] else w.loc[i, 1] for i in frente.index
    ))
    return {"sinergia": (s_partidas, s_victorias), "counter_de_a_vs_b": (c_partidas, c_victorias)}


ej_a, ej_b = "AMBER", "BROCK"
resultado_brute = validar_par(long, ej_a, ej_b)
syn_row = tablas_global["syn"].query("brawler == @ej_a and aliado == @ej_b")
print("Cálculo directo:", resultado_brute)
print("Tabla sinergia :", syn_row[["partidas", "victorias"]].values.tolist())
assert resultado_brute["sinergia"] == (int(syn_row["partidas"].iloc[0]), int(syn_row["victorias"].iloc[0])), \
    "La sinergia no coincide con el cálculo directo"
print("Validación de sinergia aprobada.")

## 8. Ejemplos de recomendación (tablas globales)
Mostramos recomendaciones para tres brawlers del alcance inicial, en ambos roles.

In [ ]:
for b in ["AMBER", "SPIKE", "MR. P"]:
    if b not in candidatos:
        continue
    print(f"\n=== {b} ===")
    print("Aliados recomendados:")
    print(recomendar(b, "aliado", tablas_global).to_string(index=False))
    print("Enemigos que lo superan (counters):")
    print(recomendar(b, "enemigo", tablas_global).to_string(index=False))

## 9. Soporte por mapa y modo
Este bloque justifica si el filtro por mapa y modo es viable con esta muestra. Para cada combinación cuenta cuántos pares de brawlers alcanzan el mínimo de partidas. Si la cifra es muy baja, las recomendaciones por mapa y modo no son confiables y conviene usar la versión global.

In [ ]:
top_mapas = long["map"].value_counts().head(3).index.tolist()
modos = long["mode"].unique().tolist()

filas = []
for mp in top_mapas:
    for md_ in modos:
        sub = long[(long["map"] == mp) & (long["mode"] == md_)]
        if sub["match_id"].nunique() == 0:
            continue
        t = build_tables(sub)
        syn_c = t["syn"][t["syn"]["brawler"].isin(candidatos) & t["syn"]["aliado"].isin(candidatos)]
        filas.append({
            "mapa": mp, "modo": md_,
            "partidas": t["n_partidas"],
            "pares_sinergia_>=min": int((syn_c["partidas"] >= MIN_MATCHES).sum()),
        })

soporte = pd.DataFrame(filas)
print(f"Mínimo de partidas configurado: {MIN_MATCHES}")
soporte

## 10. Guardado de tablas
Guardamos las tablas globales para el notebook 05, que integrará el reconocimiento visual con este motor.

In [ ]:
import os
tablas_global["syn"].to_parquet(f"{DATA_DIR}/tabla_sinergia.parquet", index=False)
tablas_global["counter"].to_parquet(f"{DATA_DIR}/tabla_counter.parquet", index=False)
tablas_global["base"].reset_index().to_csv(f"{DATA_DIR}/tabla_base.csv", index=False)

print("Archivos:", [f for f in os.listdir(DATA_DIR) if f.startswith("tabla_")])